# Customer Risk Intelligence System

## Project Overview

Modern digital platforms process millions of customer interactions every day—reviews, purchases, refunds, and support requests. Among these interactions, some may indicate operational risk such as fraudulent activity, abusive behavior, or product dissatisfaction that could lead to financial loss or reputational damage.

Traditional rule-based systems struggle to detect these patterns reliably because customer behavior is complex and constantly evolving. At the same time, fully automating decisions without transparency can create operational and regulatory concerns.

This project builds a **Customer Risk Intelligence System** that combines machine learning, decision optimization, explainable AI, and natural language generation to support intelligent and transparent decision-making.

The system predicts the **probability of risk associated with customer interactions**, evaluates the **expected business cost of different operational strategies**, and generates **clear explanations of both predictions and decisions**. The final solution is delivered through an interactive Streamlit application that enables business teams to explore and operationalize these insights.

---

## Problem Statement

Organizations need a scalable way to identify potentially risky customer interactions while balancing operational efficiency and cost. Purely manual review processes are expensive and slow, while fully automated decisions may introduce unacceptable error rates.

The key challenge is therefore:

> How can we use data and machine learning to accurately identify risky interactions while determining the most cost-effective operational response and maintaining transparency in decision making?

---

## Project Objectives

The primary objectives of this project are to:

- Develop a machine learning system that predicts the probability of risk associated with customer interactions.
- Ensure predicted probabilities are reliable through probability calibration.
- Translate model predictions into **actionable business decisions** using cost-based strategy optimization.
- Provide interpretable explanations of model behavior using explainable AI techniques.
- Generate human-readable summaries of decisions using large language models.
- Deliver the system through an interactive application that allows users to explore predictions, explanations, and recommended actions.

---

## Decision Actions

Rather than only producing predictions, the system recommends operational actions based on predicted risk and expected cost.

Three possible actions are considered:

**Approve Automatically**  
Low-risk interactions are processed automatically to maximize efficiency.

**Human Review**  
Medium-risk interactions are escalated to human analysts for verification.

**Reject / Flag**  
High-risk interactions are flagged or rejected to prevent potential loss.

The system evaluates the **expected cost of these strategies** and recommends the most economically efficient option.

---

## Key Concept Integration

This project integrates several modern AI and data science components to simulate a real industry decision system.

**Machine Learning Risk Modeling**  
Multiple models are trained to predict the probability that a customer interaction is risky.

**Probability Calibration**  
Predicted probabilities are calibrated to ensure they reflect true likelihoods and can be used reliably for decision making.

**Explainable AI (SHAP)**  
Model predictions are interpreted to understand which features most strongly influence risk assessments.

**Decision Strategy Optimization**  
Business costs associated with automation errors and human review are incorporated to determine the optimal operational strategy.

**LLM-Based Explanation Layer**  
A large language model converts technical outputs into clear explanations that business stakeholders can easily understand.

---

## Final Output and Application

The final system is deployed as an interactive **Streamlit application** that allows users to:

- View predicted customer risk scores
- Explore feature importance and model explanations
- Understand why specific decisions were recommended
- Evaluate the cost implications of different operational strategies

This type of system can be used by **risk management teams, fraud detection units, customer operations teams, and product analysts** to support faster, more transparent, and more cost-effective decision making.

Overall, the project demonstrates how modern AI techniques can be combined into an **end-to-end intelligent decision system** that transforms raw data into actionable business intelligence.

In [ ]:
# %%
# ==============================
# Core Libraries
# ==============================
import pandas as pd
import numpy as np

# ==============================
# Visualization
# ==============================
import matplotlib.pyplot as plt
import seaborn as sns

# ==============================
# Machine Learning
# ==============================
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.calibration import CalibratedClassifierCV
import xgboost as xgb

# ==============================
# NLP & Text Processing
# ==============================
import re
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from textblob import TextBlob

# ==============================
# Data Source
# ==============================
from datasets import load_dataset
from itertools import islice

# ==============================
# Utilities
# ==============================
from tqdm import tqdm
import warnings

warnings.filterwarnings('ignore')
tqdm.pandas()

import google.generativeai as genai

In [ ]:
# Reproducibility
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Display settings
pd.set_option('display.max_columns', 50)
pd.set_option('display.max_rows', 100)

 ## Basic Preprocessing Pipeline

 Define a lightweight preprocessing function to standardize raw input data.

 This step ensures:
 - Consistent text formatting
 - Basic feature creation for downstream modeling

In [ ]:
# %%
def preprocess_pipeline(df):
    df = df.copy()

    # Ensure text is string
    df['text'] = df['text'].astype(str)

    # Basic feature: review length
    df['review_length'] = df['text'].apply(lambda x: len(x.split()))

    return df

# 2. Data Acquisition

Load and prepare a structured dataset for building a **Customer Risk Prediction System**.

The dataset is sourced from **Amazon Reviews (Electronics category)** and provides:

- Customer feedback (text reviews)  
- Ratings (explicit satisfaction signals)  
- Behavioral indicators such as helpful votes and verified purchases

## Data Strategy

Due to the large scale of the dataset (millions of records), the following strategy is used:

- Use **streaming mode** to avoid downloading the entire dataset  
- Select a **domain-specific subset (Electronics)**  
- Limit the working dataset to **30,000 observations** for efficient experimentation  

This approach ensures:

- Faster iteration cycles  
- Lower memory usage  
- Sufficient data for robust modeling

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
df = pd.read_csv("amazon_reviews_sample.csv")

# Apply preprocessing
df = preprocess_pipeline(df)

df.head()


## Data Validation

 Perform initial checks to ensure data integrity before further processing.

 This includes:
 - Dataset shape
 - Data types
 - Missing values
 - Summary statistics

 ## Business Context
 The Electronics category is selected due to its strong relevance for
 customer feedback analysis and risk modeling.

 Key advantages:

 - Reviews are typically detailed and informative
 - High variation in customer satisfaction levels
 - Strong alignment with real-world e-commerce use cases

 This dataset enables:
 - Sentiment-driven customer insights
 - Behavioral risk detection
 - Decision-focused modeling for retention strategies

# 3. Data Understanding & Signal Exploration

Understand key patterns in customer behavior and identify signals relevant for risk modeling and decision-making.

Instead of performing extensive exploratory analysis, the focus is on extracting high-impact insights that directly inform:

- Feature engineering
- Risk detection logic
- Business decision strategies

## 3.1 Dataset Structure

The dataset contains customer review information including product identifiers, ratings, review metadata, and engagement signals such as helpful votes.

These variables provide behavioral indicators of customer satisfaction, engagement, and potential risk patterns.

## 3.2 Data Quality Assessment

Initial inspection evaluates:

- dataset size and dimensionality
- variable data types
- presence of missing values
- basic statistical distributions

This step ensures that the dataset is suitable for downstream preprocessing and modeling.


In [ ]:
print("Dataset Shape:", df.shape)

print("\nData Types:")
df.info()

print("\nMissing Values:")
print(df.isnull().sum())

print("\nSummary Statistics:")
df.describe()

In [ ]:
print(df.columns)

In [ ]:
df['rating'].value_counts(normalize=True)

## Core Observations

Initial exploration suggests several variables contain meaningful behavioral signals:

- Ratings capture customer satisfaction and may indicate dissatisfaction when values are low.
- Helpful votes represent community validation and may reflect the influence or credibility of reviews.
- Verified purchases provide stronger evidence that the reviewer actually bought the product.
- Review timestamps allow modeling of temporal patterns such as bursts of activity or changing sentiment.

These signals will later be transformed into structured numerical features for machine learning models. Based on these observations, the next step focuses on cleaning and transforming the data to prepare it for feature engineering and model development.

# 4. Data Cleaning & Preprocessing

Prepare the dataset for **machine learning and decision modeling** by:

- Ensuring data consistency  
- Handling missing values  
- Standardizing formats  
- Cleaning text for **NLP processing**

This step is critical for building a **reliable and reproducible data pipeline**.

In [ ]:
# Ensure essential columns are valid
df['text'] = df['text'].astype(str)

# Drop critical missing values
df = df.dropna(subset=['text', 'rating', 'user_id'])

# Re-check
df.isnull().sum()

 ## Timestamp Processing

 Convert raw timestamps into structured temporal features
 for behavioral analysis.

In [ ]:
# Convert timestamp to datetime
df['review_time'] = pd.to_datetime(df['timestamp'], unit='ms')

# Extract features
df['review_year'] = df['review_time'].dt.year
df['review_month'] = df['review_time'].dt.month

## Text Cleaning Pipeline

Clean raw review text to improve the quality of **NLP-derived features**.

The text preprocessing pipeline includes:

- Converting text to **lowercase**
- Removing noise such as **punctuation and special symbols**
- Removing common **stopwords**
- Applying **token filtering** to retain meaningful terms

In [ ]:
# Initialize stopwords
stop_words = set(ENGLISH_STOP_WORDS)

def clean_text(text, stop_words):
    text = str(text).lower()
    text = re.sub(r'[^a-z\s]', '', text)

    tokens = text.split()

    tokens = [
        word for word in tokens
        if word not in stop_words and len(word) > 2
    ]

    return " ".join(tokens)

# Apply cleaning
df['clean_text'] = df['text'].progress_apply(
    lambda x: clean_text(x, stop_words)
)

# 5 Feature Engineering

## User Behavioral Intelligence Layer

Capture user-level behavioral patterns to model trust, engagement, and influence.

Instead of treating reviews independently, this step transforms users into
economic agents with measurable behavior.

## Why This Matters

- Adds historical context to each observation  
- Enables trust-aware modeling  
- Differentiates between reliable and noisy users  

## Economic Perspective

- High average rating → lenient consumer behavior  
- High activity → experienced participant  
- High helpfulness → information contributor (opinion leader)

In [ ]:
# ==============================
# USER-LEVEL FEATURE ENGINEERING
# ==============================

def create_user_features(df):
    user_features = df.groupby('user_id').agg(
        avg_user_rating=('rating', 'mean'),
        review_frequency=('user_id', 'count'),
        avg_helpful_vote=('helpful_vote', 'mean'),
        total_helpful_votes=('helpful_vote', 'sum'),
        verified_purchase_ratio=('verified_purchase', 'mean')
    ).reset_index()

    # Avoid division issues
    user_features['helpfulness_ratio'] = (
        user_features['total_helpful_votes'] /
        user_features['review_frequency'].replace(0, 1)
    )

    return user_features[[
        'user_id',
        'avg_user_rating',
        'review_frequency',
        'avg_helpful_vote',
        'helpfulness_ratio',
        'verified_purchase_ratio'
    ]]

user_features = create_user_features(df)

df = df.merge(user_features, on='user_id', how='left')


 # 6. Target Construction & Sentiment Signals

 Construct a structured target variable and extract sentiment-based signals
 to capture customer satisfaction and dissatisfaction patterns.

 This step bridges:
 - Raw customer feedback
 - Quantifiable modeling inputs

## Binary Target Construction

Convert ratings into a **binary sentiment signal** for classification.

Target definition:

- **Positive (1):** Rating ≥ 4  
- **Negative (0):** Rating ≤ 2  
- **Neutral (3):** Removed to reduce ambiguity in model training  

This transformation ensures a **clear and robust classification framework**.

In [ ]:
# Remove neutral reviews
df = df[df['rating'] != 3]

# Binary sentiment target
df['sentiment'] = df['rating'].apply(lambda x: 1 if x >= 4 else 0)

# Distribution
df['sentiment'].value_counts()

## NLP-Based Sentiment Scoring

Extract **sentiment polarity** from review text to capture deeper behavioral signals such as:

- Emotional tone expressed in the review
- Hidden dissatisfaction not reflected in ratings
- Mismatch between rating score and actual sentiment

This step is critical for detecting **information asymmetry** and identifying **high-risk behavior**.

In [ ]:
def get_sentiment(text):
    return TextBlob(text).sentiment.polarity

df['sentiment_score'] = df['clean_text'].apply(get_sentiment)

In [ ]:

df['sentiment_score'].describe()

## Class Imbalance Strategy

The dataset naturally contains more positive reviews than negative ones.

This reflects real-world marketplaces where satisfied users dominate.

Instead of artificially balancing the data, we adopt a realistic strategy:

- Use class-weighted models  
- Focus on recall for negative class  
- Evaluate using F1-score and ROC-AUC  

## 6.5 Business Implication

Negative reviews represent:

- Customer dissatisfaction  
- Product failure signals  
- Revenue and reputation risk  

Detecting them accurately is more valuable than overall accuracy.


## 7. Risk Intelligence Layer (Information Asymmetry Modeling)

Model hidden risks in reviews using behavioral inconsistencies and
information asymmetry principles.

Not all reviews are equally trustworthy — this layer identifies
low-quality, inconsistent, or potentially misleading signals.

## Conceptual Foundation

Based on economic theory:

- Adverse Selection → unreliable participants entering system  
- Moral Hazard → inconsistent behavior after participation  
- Information Asymmetry → unequal access to truthful information  

## 7.3 Why This is Important (Industry Relevance)

This layer extends beyond traditional sentiment analysis by incorporating behavioral and trust-based signals.

This layer enables:

- Trust scoring systems  
- Fraud detection signals  
- Risk-aware recommendation engines  
- Decision-support systems  

In [ ]:
# ==============================
# RISK FEATURE ENGINEERING
# ==============================

# Step 1: Rating–Sentiment Mismatch

df['normalized_rating'] = df['rating'] / 5
df['normalized_sentiment'] = (df['sentiment_score'] + 1) / 2

df['rating_sentiment_gap'] = abs(
    df['normalized_rating'] - df['normalized_sentiment']
)


# Step 2: Extreme Rating Behavior

df['is_extreme_rating'] = df['rating'].apply(
    lambda x: 1 if x in [1, 5] else 0
)


# Step 3: Behavioral Instability

df['rating_deviation'] = abs(
    df['rating'] - df['avg_user_rating']
)


# Step 4: Low Credibility Signal

df['low_helpfulness_flag'] = df['helpful_vote'].apply(
    lambda x: 1 if x == 0 else 0
)


# Step 5: Composite Risk Score (Weighted)

df['risk_score'] = (
    df['rating_sentiment_gap'] * 0.4 +
    df['rating_deviation'] * 0.3 +
    df['low_helpfulness_flag'] * 0.2 +
    df['is_extreme_rating'] * 0.1
)


## Risk Classification

Convert continuous risk score into interpretable business categories.

This is critical for decision-making systems and dashboards.

In [ ]:
HIGH_RISK_THRESHOLD = 0.6
MEDIUM_RISK_THRESHOLD = 0.3

def classify_risk(score):
    if score > HIGH_RISK_THRESHOLD:
        return 'High'
    elif score > MEDIUM_RISK_THRESHOLD:
        return 'Medium'
    return 'Low'

df['risk_level'] = df['risk_score'].apply(classify_risk)

df[['rating', 'sentiment_score', 'risk_score', 'risk_level']].head()

In [ ]:
df['risk_level'].value_counts()

## 8. Customer Intelligence & Segmentation Layer

## 8.1 Objective

Transform user behavior into actionable customer segments
that businesses can directly use.

This shifts the system from:

Data Analysis → Decision Intelligence

## 8.2 Why This Matters

Businesses do not act on individual reviews.

They act on:

- Customer types  
- Behavioral patterns  
- Risk profiles  

## 8.3 Segmentation Dimensions

Users are segmented based on:

1. Engagement → activity level  
2. Satisfaction → average rating  
3. Risk → behavioral inconsistency  

## 8.4 Business Output

Segments include:

- High-Value Customers  
- Risky Users  
- Critical Users  
- Passive Users  
- Regular Users  
These segments provide an interpretable behavioral layer that can support
risk-aware decision systems, targeted interventions, and customer retention strategies.

In [ ]:
# %%
# ==============================
# USER SEGMENTATION
# ==============================

user_segmentation = df.groupby('user_id').agg(
    avg_satisfaction=('rating', 'mean'),
    engagement_level=('text', 'count'),
    avg_risk=('risk_score', 'mean')
).reset_index()


def segment_user(row):

    if row['avg_satisfaction'] >= 4 and row['avg_risk'] < 0.3 and row['engagement_level'] >= 3:
        return 'High-Value'

    elif row['avg_risk'] > 0.6:
        return 'Risky'

    elif row['avg_satisfaction'] < 3:
        return 'Critical'

    elif row['engagement_level'] < 2:
        return 'Passive'

    else:
        return 'Regular'


user_segmentation['customer_segment'] = user_segmentation.apply(segment_user, axis=1)


df = df.merge(
    user_segmentation[['user_id', 'customer_segment']],
    on='user_id',
    how='left'
)

In [ ]:
user_segmentation['customer_segment'].value_counts()

# 9. Risk Label Construction & Validation

The goal of this step is to translate raw review behavior into a **business-relevant risk signal**.  
Instead of predicting ratings directly, we construct a **risk label** that captures patterns associated with potential operational or reputational issues.

This aligns with real-world analytics use cases where companies are less interested in average performance and more focused on **identifying problematic transactions early**.

## Defining the Risk Label

A review is classified as **risky** if it reflects negative customer experience or signals potential friction in operations.

We define the risk label using the following rule:

- Reviews with **rating ≤ 2 → Risky (1)**
- Reviews with **rating > 2 → Not Risky (0)**


In [ ]:
# ==============================
# Risk Label Construction
# ==============================

# Condition 1: Low rating
low_rating = df['rating'] <= 2

# Condition 2: Negative sentiment
negative_sentiment = df['sentiment_score'] < 0

# Condition 3: Rating-Sentiment mismatch
inconsistency = df['rating_sentiment_gap'] > 0.4

# Condition 4: Low helpfulness
low_helpfulness = df['helpfulness_ratio'] < 0.1

# Condition 5: High behavioral risk
high_risk_score = df['risk_score'] > 0.6

# Combined logic
df['risk_label'] = (
    (low_rating & negative_sentiment) |
    (inconsistency & low_helpfulness) |
    (high_risk_score)
).astype(int)

# Distribution check
df['risk_label'].value_counts(normalize=True)

In [ ]:
# ==============================
# Risk Label Validation
# ==============================

# Compare behavior across labels
df.groupby('risk_label')[[
    'rating',
    'sentiment_score',
    'helpfulness_ratio',
    'risk_score'
]].mean()

In [ ]:
# Class balance
df['risk_label'].value_counts()

In [ ]:
# ==============================
# Feature Selection
# ==============================

feature_cols = [
    'rating',
    'sentiment_score',
    'verified_purchase',
    'review_length',
    'helpfulness_ratio'
]

X = df[feature_cols]
y = df['risk_label']

# 10 Economic Cost Layer (Order Value Integration)

In real-world systems, not all transactions have the same financial impact.

A $10 order and a $500 order should not be treated equally in decision-making.

To address this, we introduce **order-level monetary value**, which allows the system to:

- Scale risk decisions based on financial exposure
- Improve realism of cost-sensitive decision systems
- Align with production-grade e-commerce logic

---

### Data Source

We use product metadata (price) as a proxy for order value.

Since raw price data is noisy and inconsistent, we apply:

- Text cleaning
- Numeric extraction
- Range averaging
- Missing value imputation

---

### Business Impact

This enhancement transforms the system from:

Static Rule System → **Dynamic Economic Decision Engine**

This is a key differentiator in real-world AI systems.

In [ ]:
# ==============================
# LOAD METADATA (PRICE INFO)
# ==============================

from google.colab import files
uploaded = files.upload()

In [ ]:
meta_df = pd.read_csv("amazon_meta_sample.csv")

meta_df.head()

In [ ]:
print("Metadata Columns:")
print(meta_df.columns)

In [ ]:
# ==============================
# KEEP REQUIRED COLUMNS
# ==============================

meta_df = meta_df[['parent_asin', 'price']]

In [ ]:
import re

def clean_price(price):
    if pd.isna(price):
        return np.nan

    price = str(price)
    numbers = re.findall(r'\d+\.?\d*', price)

    if len(numbers) == 0:
        return np.nan

    numbers = list(map(float, numbers))
    return sum(numbers) / len(numbers)

meta_df['price'] = meta_df['price'].apply(clean_price)
meta_df = meta_df.drop_duplicates(subset='parent_asin')

In [ ]:
# ==============================
# MERGE USING PRODUCT FAMILY KEY
# ==============================

df = df.merge(
    meta_df,
    left_on='asin',
    right_on='parent_asin',
    how='left'
)

In [ ]:
# Drop extra column
df = df.drop(columns=['parent_asin'])

# Fill missing prices
df['price'] = df['price'].fillna(df['price'].median())

# Create order value
df['order_value'] = df['price'].clip(lower=5, upper=500)

In [ ]:
df[['asin', 'price', 'order_value']].head()
df['order_value'].describe()

In [ ]:
print(df.columns)

In [ ]:
# ==============================
# Separate Features
# ==============================

X = df[[
    'rating',
    'sentiment_score',
    'review_length',
    'verified_purchase',
    'helpfulness_ratio'
]]
y = df['risk_label']

# Keep order_value separately
order_values = df['order_value']

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test, ov_train, ov_test = train_test_split(
    X, y, order_values,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [ ]:
# %%
# ==============================
# Feature Scaling
# ==============================

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

# Fit ONLY on training data (important!)
X_train_scaled = scaler.fit_transform(X_train)

# Transform test data
X_test_scaled = scaler.transform(X_test)

print("Scaling completed.")

# 11 Model Training and Evaluation

## Logistic Regression (Baseline Model)

Train a baseline classification model to predict whether a review is **risky (1)** or **safe (0)**.

### Why Logistic Regression?

- Industry-standard **baseline model**
- Highly **interpretable**, as coefficients explain feature impact
- **Fast and computationally efficient**

This model establishes a **reference performance benchmark** before moving to more complex models.

### Key Considerations

- Use **scaled features** generated with `StandardScaler`
- Apply **class balancing** to address class imbalance (~30% risky cases)
- The model outputs both:

  - **Predictions** for classification
  - **Probabilities** for business decision-making

In [ ]:
# %%
# ==============================
# Logistic Regression Model
# ==============================

from sklearn.linear_model import LogisticRegression

log_model = LogisticRegression(
    class_weight='balanced',   # handle class imbalance
    random_state=42,
    max_iter=1000
)

log_model.fit(X_train_scaled, y_train)

In [ ]:
# %%
# ==============================
# Predictions
# ==============================

y_pred_log = log_model.predict(X_test_scaled)
y_prob_log = log_model.predict_proba(X_test_scaled)[:, 1]

## Model Evaluation

The model is evaluated using the following metrics:

- **Precision, Recall, and F1-score** → Measure classification quality  
- **ROC-AUC Score** → Measures the model's ability to distinguish between risky and safe users  

These metrics are particularly important in business settings where:

- Missing a risky customer (**false negative**) can be costly  
- Over-flagging safe users (**false positive**) may negatively affect customer experience

In [ ]:
# %%
# ==============================
# Model Evaluation
# ==============================

from sklearn.metrics import classification_report, roc_auc_score

print("Classification Report:\n")
print(classification_report(y_test, y_pred_log))

roc_auc = roc_auc_score(y_test, y_prob_log)
print("\nROC-AUC Score:", roc_auc)

## Feature Importance (Interpretability)

Logistic Regression provides **model coefficients** that indicate feature influence:

- **Positive coefficient** → Increases the probability of risk  
- **Negative coefficient** → Decreases the probability of risk  

This interpretability is valuable for:

- **Business explainability**
- **Stakeholder communication**
- Identifying key **risk drivers**

In [ ]:
# ==============================
# Feature Importance (Coefficients)
# ==============================

coefficients = pd.DataFrame({
    'Feature': feature_cols,
    'Coefficient': log_model.coef_[0]
}).sort_values(by='Coefficient', ascending=False)

coefficients

## Model Interpretation (Logistic Regression)

### Key Insights

- The model achieves a strong **ROC-AUC (~0.79)**, indicating good ability to distinguish between risky and safe reviews
- **Recall for the risky class (68%)** is prioritized, aligning with the business objective of minimizing missed risks
- Some **false positives** occur, but this is acceptable in risk detection systems

---

### Feature-Level Insights

- Higher **ratings** and **sentiment scores** significantly reduce risk probability
- **Longer reviews** and **helpful votes** are associated with lower risk
- **Verified purchases** show slightly higher risk, possibly reflecting stricter customer expectations

---

### Business Interpretation

The model captures several important signals:

- Customer dissatisfaction patterns  
- Behavioral inconsistencies  
- Information reliability  

This makes it suitable as a baseline **risk detection engine** for:

- Customer experience monitoring  
- Fraud or anomaly detection  
- Review quality assessment

# Random Forest Model (Non-Linear Baseline)

Improve predictive performance by capturing **non-linear relationships** and **interactions** between behavioral and textual features.


## Why Random Forest?

- Handles **non-linearity** automatically  
- Captures **feature interactions**  
- Robust to **noise and outliers**  
- Does **not require feature scaling**


## Business Perspective

Unlike **Logistic Regression (linear)**, Random Forest can model:

- Complex **user behavior patterns**
- Hidden interactions between **sentiment and ratings**
- **Non-linear risk signals**

This makes the model more suitable for **real-world customer analytics systems**.

In [ ]:
# ==============================
# Random Forest Model
# ==============================

from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=8,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

rf_model.fit(X_train, y_train)

In [ ]:
# ==============================
# Predictions
# ==============================

y_pred_rf = rf_model.predict(X_test)
y_prob_rf = rf_model.predict_proba(X_test)[:, 1]

In [ ]:
# ==============================
# Model Evaluation
# ==============================

from sklearn.metrics import classification_report, roc_auc_score

print("Classification Report:\n")
print(classification_report(y_test, y_pred_rf))

roc_auc_rf = roc_auc_score(y_test, y_prob_rf)
print("\nROC-AUC Score:", roc_auc_rf)

In [ ]:
# ==============================
# Feature Importance
# ==============================

rf_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': rf_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

rf_importance

## Model Interpretation (Random Forest)

### Key Insights

- The model shows a strong improvement in overall performance with **ROC-AUC (~0.89)**
- **Precision for the risky class increases significantly**, reducing false positives
- **Recall remains stable**, indicating consistent ability to detect risky users

---

### Comparison with Logistic Regression

- Random Forest captures **non-linear relationships and feature interactions**
- Provides **higher overall accuracy and stability**
- Trades a small drop in recall for a **large gain in precision**

---

### Feature-Level Insights

- **Rating and sentiment** dominate as primary risk indicators
- **Helpfulness ratio** contributes as a trust signal
- **Verified purchase** and **review length** provide secondary behavioral context

---

### Business Interpretation

The model is better suited for:

- Reducing **false alarms** in risk detection systems
- Improving reliability of **automated decision pipelines**
- Supporting **production-level customer risk monitoring**

# XGBoost Model (Industry-Level Gradient Boosting)

Further improve predictive performance using **advanced boosting techniques** that sequentially learn from previous prediction errors.

## Why XGBoost?

- Industry-standard model for **tabular data**
- Handles **non-linear relationships and complex feature interactions**
- Built-in **regularization** helps reduce overfitting
- Often achieves **state-of-the-art performance**

## Business Perspective

XGBoost enables:

- More **accurate risk prediction**
- Better identification of **subtle behavioral patterns**
- **Production-grade performance** for decision systems

In [ ]:
# ==============================
# XGBoost Model
# ==============================

from xgboost import XGBClassifier

xgb_model = XGBClassifier(
    n_estimators=200,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight= (len(y_train[y_train==0]) / len(y_train[y_train==1])),
    random_state=42,
    use_label_encoder=False,
    eval_metric='logloss'
)

xgb_model.fit(X_train, y_train)

In [ ]:
# ==============================
# Predictions
# ==============================

y_pred_xgb = xgb_model.predict(X_test)
y_prob_xgb = xgb_model.predict_proba(X_test)[:, 1]

In [ ]:
# ==============================
# Model Evaluation
# ==============================

print("Classification Report:\n")
print(classification_report(y_test, y_pred_xgb))

roc_auc_xgb = roc_auc_score(y_test, y_prob_xgb)
print("\nROC-AUC Score:", roc_auc_xgb)

In [ ]:
# ==============================
# Feature Importance
# ==============================

xgb_importance = pd.DataFrame({
    'Feature': feature_cols,
    'Importance': xgb_model.feature_importances_
}).sort_values(by='Importance', ascending=False)

xgb_importance


 # 9.4 Model Comparison and Selection

 Compare all models to identify the best-performing approach
 for predicting customer risk in a business setting.

 ## Models Evaluated

 - Logistic Regression (Baseline)
 - Random Forest (Non-linear model)
 - XGBoost (Advanced boosting model)

In [ ]:
# ==============================
# Model Comparison Table
# ==============================

comparison_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost'],
    'Accuracy': [0.74, 0.82, 0.84],
    'ROC_AUC': [0.79, 0.886, 0.908],
    'Precision_Risk': [0.57, 0.73, 0.72],
    'Recall_Risk': [0.68, 0.67, 0.77]
})

comparison_df

## Performance Summary

- **XGBoost** achieves the highest overall performance across most evaluation metrics.
- **Random Forest** provides a strong balance between precision and model stability.
- **Logistic Regression** serves as a useful baseline with high interpretability.

## Key Trade-offs

### Logistic Regression
- Better **recall** than Random Forest  
- Lower **overall predictive performance**

### Random Forest
- Higher **precision**
- Slightly lower **recall** for risky users

### XGBoost
- Best balance between **recall**, **precision**, and **overall accuracy**

## Model Selection

**XGBoost is selected as the final model because:**

- It achieves the **highest ROC-AUC** (strongest ranking ability)
- Maintains **strong recall for risky users**, which is critical for business applications
- It is **robust and scalable for production environments**


## Business Interpretation

The selected model enables:

- **Early identification** of high-risk customers
- Improved **decision-making** in customer management
- Reduction in **operational and reputational risks**

# Probability Calibration (Decision Reliability Layer)

Ensure that predicted probabilities reflect the **true likelihood of risk**.

In real-world decision systems, probabilities are not only used for ranking —  
they directly influence **financial and operational decisions**.

Example:

- A predicted **0.80 risk probability** should correspond to approximately **80% actual risk**.

Without calibration:

- Models may become **overconfident or underconfident**
- **Cost optimization** becomes unreliable

## Why Calibration Matters

Probability calibration:

- Converts model outputs into **decision-grade probabilities**
- Is critical for:

  - **Cost-sensitive learning**
  - **Risk thresholding**
  - **Business decision engines**

## Method Used

We apply **Platt Scaling (Sigmoid Calibration)**:

- Works well across many classification models
- Suitable for **small to medium datasets**
- Serves as a strong **industry-standard baseline**

In [ ]:
# ==============================
# Calibrated Models
# ==============================

# Logistic Regression Calibration
calibrated_log = CalibratedClassifierCV(
    estimator=log_model,
    method='sigmoid',
    cv=5
)

calibrated_log.fit(X_train_scaled, y_train)


# Random Forest Calibration
calibrated_rf = CalibratedClassifierCV(
    estimator=rf_model,
    method='sigmoid',
    cv=5
)

calibrated_rf.fit(X_train, y_train)


# XGBoost Calibration
calibrated_xgb = CalibratedClassifierCV(
    estimator=xgb_model,
    method='sigmoid',
    cv=5
)

calibrated_xgb.fit(X_train, y_train)

In [ ]:
# ==============================
# Calibrated Probabilities
# ==============================

y_prob_log_cal = calibrated_log.predict_proba(X_test_scaled)[:, 1]
y_prob_rf_cal = calibrated_rf.predict_proba(X_test)[:, 1]
y_prob_xgb_cal = calibrated_xgb.predict_proba(X_test)[:, 1]

In [ ]:
# ==============================
# ROC-AUC After Calibration
# ==============================

from sklearn.metrics import roc_auc_score

print("Logistic Regression (Calibrated) ROC-AUC:",
      roc_auc_score(y_test, y_prob_log_cal))

print("Random Forest (Calibrated) ROC-AUC:",
      roc_auc_score(y_test, y_prob_rf_cal))

print("XGBoost (Calibrated) ROC-AUC:",
      roc_auc_score(y_test, y_prob_xgb_cal))

In [ ]:
# ==============================
# Calibration Curve
# ==============================

from sklearn.calibration import calibration_curve

def plot_calibration(y_true, y_prob, model_name):
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=10)

    plt.figure()
    plt.plot(prob_pred, prob_true, marker='o')
    plt.plot([0, 1], [0, 1], linestyle='--')

    plt.title(f'Calibration Curve - {model_name}')
    plt.xlabel('Predicted Probability')
    plt.ylabel('True Probability')

    plt.show()


# Plot for each model
plot_calibration(y_test, y_prob_log_cal, "Logistic Regression")
plot_calibration(y_test, y_prob_rf_cal, "Random Forest")
plot_calibration(y_test, y_prob_xgb_cal, "XGBoost")

The calibration curve indicates that predicted probabilities closely match observed event frequencies, confirming that the calibrated model provides reliable probability estimates for decision optimization.

In [ ]:
# ==============================
# Calibration Evaluation — Brier Score
# ==============================

from sklearn.metrics import brier_score_loss

brier = brier_score_loss(y_test, y_prob_xgb_cal)

print("Brier Score:", round(brier, 4))

# Model Interpretability using SHAP

## Objective

In this section, we interpret the predictions of the final calibrated model using SHAP (SHapley Additive exPlanations).

While previous steps focused on predictive performance (ROC-AUC) and probability calibration, this step focuses on understanding why the model assigns risk to specific observations.

This is critical for:

- Explaining model behavior to stakeholders  
- Identifying key drivers of customer risk  
- Supporting economically rational decision-making  
- Enabling targeted business interventions  

## Why SHAP?

SHAP provides a unified framework to:

- Measure feature importance (global interpretability)  
- Explain individual predictions (local interpretability)  

Unlike traditional feature importance methods, SHAP ensures consistent and theoretically grounded explanations based on contribution of each feature to the prediction.

## Link to Decision Engine

The outputs from SHAP will be directly used in the next section (Decision Engine) to:

- Design targeted actions based on *drivers* of risk  
- Improve cost-efficiency of interventions  
- Move from generic decisions to personalized strategies  


In [ ]:
# Initialize SHAP Explainer for XGBoost
explainer = shap.TreeExplainer(xgb_model)

# Generate SHAP values
shap_values_test = explainer(X_test)

In [ ]:
# Global Feature Importance Plot
shap.summary_plot(shap_values_test, X_test)

## SHAP Summary Plot — Global Feature Impact

This plot shows how each feature influences the model’s predictions across all customers, with features ranked by importance from top to bottom. The horizontal position indicates whether a feature increases (right) or decreases (left) predicted risk, while color represents feature value (red = high, blue = low). The model is primarily driven by `rating` and `sentiment_score`, where low ratings and negative sentiment strongly increase risk, and high values reduce it. `helpfulness_ratio` also plays a meaningful role, while `verified_purchase` and `review_length` have smaller but consistent effects. Overall, the model relies heavily on customer satisfaction signals, which aligns well with real-world expectations.

In [ ]:
# Bar plot for feature importance
shap.plots.bar(shap_values_test)

## Feature Importance Ranking (Mean SHAP Values)

This bar chart summarizes the average impact of each feature on model predictions, showing that `rating` is the most influential factor, followed by `sentiment_score` and `helpfulness_ratio`. The remaining features, `verified_purchase` and `review_length`, contribute less but still provide additional context. This indicates that the model prioritizes direct indicators of customer experience (ratings and sentiment) over secondary behavioral features, confirming that the predictive structure is both logical and aligned with business intuition.

In [ ]:
# Select a single observation
sample_index = 0

# Waterfall plot for individual prediction
shap.plots.waterfall(shap_values_test[sample_index])

## Individual Prediction Explanation (Waterfall Plot)

This plot explains how different features contributed to the prediction for a single customer by starting from a baseline risk and adjusting it based on feature values. In this case, `sentiment_score`, `verified_purchase`, `helpfulness_ratio`, and a high `rating` all push the prediction toward lower risk, while `review_length` has a very small positive contribution to risk. The combined effect results in a low overall predicted risk for this customer, demonstrating how multiple positive signals together reduce risk and showing how the model makes decisions at an individual level.

# 10. Decision Strategy Engine – AI vs Human Cost Optimization

In production systems, machine learning models **do not directly make business decisions**.  
Instead, they produce **risk probabilities**, which must be translated into **operational strategies**.

In e-commerce environments (such as refund or complaint handling systems), organizations must decide **how each case should be processed**:

- Should it be handled automatically by AI?
- Should it be escalated to a human agent?
- Or should the system use a hybrid approach?

This section upgrades the previous decision engine into a **strategy optimization system** that evaluates the **expected financial impact of three operational strategies**.

---

## Operational Strategies Evaluated

### Strategy A — AI Automation

All cases are processed automatically by AI systems.

Advantages:

- Very low operational cost
- Extremely scalable

Risks:

- AI may incorrectly approve fraudulent cases

---

### Strategy B — Human Review

Every case is reviewed by a human agent.

Advantages:

- Higher decision accuracy
- Lower fraud risk

Costs:

- Operational labor cost
- Slower processing

---

### Strategy C — Hybrid AI + Human

AI handles low-risk cases automatically while high-risk cases are escalated to human review.

Advantages:

- Maintains scalability
- Reduces fraud risk
- Balances cost and accuracy

---

The system evaluates the **expected cost of each strategy** using predicted fraud probability and order value, and selects the **strategy that minimizes expected financial loss**.

In [ ]:
# ==============================
# Prepare Prediction Data
# ==============================

# Use calibrated probabilities (IMPORTANT UPGRADE)
y_prob = calibrated_xgb.predict_proba(X_test)[:, 1]

decision_df = X_test.copy()
decision_df['actual_label'] = y_test.values
decision_df['risk_probability'] = y_prob

# IMPORTANT: bring order_value into decision layer
decision_df['order_value'] = ov_test.values

decision_df.head()

## Strategy Cost Parameters

The decision engine models the expected financial impact of different handling strategies using a set of configurable cost parameters.

### Fraud Loss

If a fraudulent transaction is incorrectly approved, the financial loss is proportional to the transaction value.

Fraud Loss = FRAUD_COST_FACTOR × order_value

Where:

- `order_value` represents the financial exposure of the transaction.
- `FRAUD_COST_FACTOR` allows the system to scale fraud impact if additional costs (such as logistics or chargeback fees) are considered.

For this simulation:

FRAUD_COST_FACTOR = 4.0

---

### Human Review Cost

Human investigation introduces a fixed operational cost that reflects labor and processing overhead.

C_review = 4

---

### Automation Error Rate

Automated systems may occasionally misclassify risky transactions.

AI_error_rate = 0.30

---

### Human Error Rate

Human reviewers typically make fewer mistakes, but errors are still possible.

Human_error_rate = 0.05

## Strategy Cost Simulation

For each transaction, the system estimates the expected cost of three operational strategies.

Let:

- `p` = predicted fraud probability
- `v` = order value

---

### Strategy A — AI Automation

All cases handled automatically.

Expected cost arises only if AI incorrectly approves fraudulent transactions.

E(cost_AI) = p × v × AI_error_rate

---

### Strategy B — Human Review

Every transaction is reviewed by a human agent.

Expected cost includes both the review cost and a small chance of human error.

E(cost_Human) = review_cost + p × v × human_error_rate

---

### Strategy C — Hybrid

AI processes low-risk cases automatically while high-risk cases are escalated to human review.

This strategy attempts to balance operational cost and fraud prevention.

In [ ]:
# ==============================
# Strategy Cost Functions
# ==============================

def cost_ai_strategy(p, order_value):

    fraud_loss = FRAUD_COST_FACTOR * order_value

    return p * fraud_loss * AI_ERROR_RATE


def cost_human_strategy(p, order_value):

    fraud_loss = FRAUD_COST_FACTOR * order_value

    return HUMAN_REVIEW_COST + (p * fraud_loss * HUMAN_ERROR_RATE)


HYBRID_THRESHOLD = 0.35

def cost_hybrid_strategy(p, order_value):

    fraud_loss = FRAUD_COST_FACTOR * order_value

    if p < HYBRID_THRESHOLD:
        return p * fraud_loss * AI_ERROR_RATE

    else:
        return HUMAN_REVIEW_COST + (p * fraud_loss * HUMAN_ERROR_RATE)

In [ ]:
# ==============================
# Strategy Cost Parameters
# ==============================

# Fraud loss is proportional to transaction value
FRAUD_COST_FACTOR = 3.0

# Operational cost of manual investigation
HUMAN_REVIEW_COST = 4

# Estimated error rates
AI_ERROR_RATE = 0.30
HUMAN_ERROR_RATE = 0.05

print("Strategy Cost Setup")
print("Fraud Cost Factor:", FRAUD_COST_FACTOR)
print("Human Review Cost:", HUMAN_REVIEW_COST)
print("AI Error Rate:", AI_ERROR_RATE)
print("Human Error Rate:", HUMAN_ERROR_RATE)

In [ ]:
# ==============================
# Compute Strategy Costs
# ==============================

decision_df['cost_ai'] = decision_df.apply(
    lambda x: cost_ai_strategy(x['risk_probability'], x['order_value']), axis=1
)

decision_df['cost_human'] = decision_df.apply(
    lambda x: cost_human_strategy(x['risk_probability'], x['order_value']), axis=1
)

decision_df['cost_hybrid'] = decision_df.apply(
    lambda x: cost_hybrid_strategy(x['risk_probability'], x['order_value']), axis=1
)

decision_df[['risk_probability','order_value','cost_ai','cost_human','cost_hybrid']].head()

In [ ]:
# ==============================
# Select Optimal Strategy
# ==============================

def choose_strategy(row):

    costs = {
        "AI Automation": row["cost_ai"],
        "Human Review": row["cost_human"],
        "Hybrid AI + Human": row["cost_hybrid"]
    }

    return min(costs, key=costs.get)


decision_df["optimal_strategy"] = decision_df.apply(choose_strategy, axis=1)

decision_df["expected_cost"] = decision_df[
    ["cost_ai","cost_human","cost_hybrid"]
].min(axis=1)

## Optimal Action Selection — Interpretation

The model selects the action with the lowest expected cost for each transaction. High-risk cases are consistently rejected, while low-risk cases are approved. This demonstrates that the decision engine is functioning as intended, translating predicted probabilities into economically rational decisions rather than relying on arbitrary thresholds.

## SHAP Integration — Explaining Decisions

While the model provides **risk probabilities**, business decisions also require understanding **why** a transaction is considered risky.

To achieve this, we integrate **SHAP values** into the decision layer to identify the **key drivers behind each prediction**.

This enables the system to:

- Provide **transparency** into model decisions
- Identify **root causes of risk**
- Enable **targeted business interventions**

For each transaction, we extract the **top contributing features** influencing the prediction.

In [ ]:
# ==============================
# SHAP Integration
# ==============================

# Convert SHAP values for TEST data into DataFrame
shap_df = pd.DataFrame(
    shap_values_test.values,
    columns=X_test.columns,
    index=X_test.index
)

# Rename columns to avoid collision
shap_df = shap_df.add_prefix("shap_")

# ======================================
# Attach SHAP explanations to decisions
# ======================================

decision_df = decision_df.join(shap_df)

In [ ]:
decision_df.head()

## SHAP-Based Decision Explanation — Interpretation

The inclusion of top risk drivers provides transparency into each decision by highlighting the most influential features. For rejected transactions, factors such as low ratings and negative sentiment are dominant, while approved transactions are driven by positive signals like higher helpfulness or sentiment. This enhances trust in the system by linking decisions to interpretable customer behavior patterns.

## Final Decision Output

The final system combines:

- **Predicted risk probability**
- **Cost-based optimization**
- **Feature-level explanations**

Each decision now includes:

- **Optimal action** (approve, reject, review)
- **Expected financial cost**
- **Key factors driving the decision**

This transforms the model from a **prediction tool** into a **decision intelligence system**.

---

## Business Value

This approach enables:

- **Transparent decision-making**
- Improved **trust in AI systems**
- **Actionable insights** for operations teams
- Better alignment between **data science and business strategy**

**Example**

A transaction may be flagged due to **low rating and negative sentiment**, allowing **targeted intervention** instead of blanket rejection.

In [ ]:
# ==============================
# SHAP-Based Decision Explanation
# ==============================

def extract_top_drivers(row, feature_cols, top_n=3):
    """
    Identify the top SHAP drivers for a prediction.
    """

    shap_values_row = row[feature_cols]

    # Absolute importance ranking
    top_features = shap_values_row.abs().sort_values(ascending=False).head(top_n)

    return ", ".join(top_features.index)


# Identify SHAP feature columns
shap_feature_cols = list(X_test.columns)

# Extract top drivers
decision_df["top_risk_drivers"] = decision_df.apply(
    lambda row: extract_top_drivers(row, shap_feature_cols),
    axis=1
)

In [ ]:
# ==============================
# Decision Explanation Text
# ==============================

def generate_decision_explanation(row):

    strategy = row["optimal_strategy"]
    drivers = row["top_risk_drivers"]
    prob = row["risk_probability"]

    if strategy == "AI Automation":
        return f"AI automation recommended due to low predicted risk ({prob:.2f}). Key signals: {drivers}."

    elif strategy == "Human Review":
        return f"Human review recommended due to elevated risk ({prob:.2f}). Key drivers: {drivers}."

    else:
        return f"Hybrid handling recommended: AI for low-risk processing with escalation for high risk ({prob:.2f}). Key drivers: {drivers}."

In [ ]:
# ==============================
# Generate Decision Explanation
# ==============================

decision_df["decision_explanation"] = decision_df.apply(
    generate_decision_explanation,
    axis=1
)

decision_df.head()

In [ ]:
decision_df.to_csv("decision_engine_output.csv", index=False)

In [ ]:
from google.colab import files
files.download("decision_engine_output.csv")

In [ ]:
# ==============================
# Action Distribution
# ==============================

decision_distribution = decision_df['optimal_strategy'].value_counts(normalize=True)

print("Decision Distribution:\n")
print(decision_distribution)

In [ ]:
# %% [markdown]
# # 13. Policy Simulation – Strategy Evaluation & Business Impact
#
# In production environments, decision models are rarely deployed with fixed assumptions.
# Organizations continuously evaluate operational strategies to understand how different
# policies affect financial outcomes, risk exposure, and system efficiency.
#
# While the decision engine determines the optimal strategy for each transaction,
# the **simulation layer evaluates how the overall system performs under realistic conditions**.
#
# ---
#
# ## Objective
#
# The goal of this simulation is to evaluate the **financial impact of the AI decision system**
# using realistic operational assumptions.
#
# Specifically, we measure:
#
# - Total realized cost across all transactions
# - Average cost per order
# - Distribution of operational strategies
# - Cost breakdown across decision types
#
# ---
#
# ## Operational Strategies Evaluated
#
# The decision engine assigns one of three strategies to each transaction:
#
# **AI Automation**
# - Fully automated handling
# - Lowest operational cost
# - Higher risk of automation errors
#
# **Human Review**
# - Manual investigation
# - Higher operational cost
# - Lower error probability
#
# **Hybrid AI + Human**
# - AI handles low-risk cases automatically
# - High-risk cases escalate to human review
# - Balances cost efficiency and fraud protection
#
# ---
#
# ## Why Simulation Matters
#
# Predictive models estimate risk probabilities, but business leaders must understand
# **how those predictions translate into financial outcomes**.
#
# This simulation therefore evaluates:
#
# Risk Prediction → Operational Strategy → Financial Impact
#
# By incorporating **true labels from the test data**, we can estimate the
# **realized cost of decisions**, providing a realistic approximation of
# how the system would perform in production.

In [ ]:
# %%
# ==============================
# Policy Simulation Function
# ==============================

def run_policy_simulation(df):

    sim_df = df.copy()

    # ---------------------------------
    # Step 1: Strategy Cost Calculations
    # ---------------------------------

    sim_df["cost_ai"] = (
        sim_df["risk_probability"]
        * (FRAUD_COST_FACTOR * sim_df["order_value"])
        * AI_ERROR_RATE
    )

    sim_df["cost_human"] = (
        HUMAN_REVIEW_COST
        + sim_df["risk_probability"]
        * (FRAUD_COST_FACTOR * sim_df["order_value"])
        * HUMAN_ERROR_RATE
    )

    sim_df["cost_hybrid"] = sim_df.apply(
        lambda row:
        row["risk_probability"]
        * (FRAUD_COST_FACTOR * row["order_value"])
        * AI_ERROR_RATE
        if row["risk_probability"] < HYBRID_THRESHOLD
        else HUMAN_REVIEW_COST
        + row["risk_probability"]
        * (FRAUD_COST_FACTOR * row["order_value"])
        * HUMAN_ERROR_RATE,
        axis=1
    )

    # ---------------------------------
    # Step 2: Strategy Selection
    # ---------------------------------

    def choose_strategy(row):

        costs = {
            "AI Automation": row["cost_ai"],
            "Human Review": row["cost_human"],
            "Hybrid AI + Human": row["cost_hybrid"]
        }

        return min(costs, key=costs.get)

    sim_df["optimal_strategy"] = sim_df.apply(choose_strategy, axis=1)

    # ---------------------------------
    # Step 3: Realized Cost Calculation
    # ---------------------------------

    def realized_cost(row):

        strategy = row["optimal_strategy"]
        label = row["actual_label"]
        value = row["order_value"]

        fraud_loss = FRAUD_COST_FACTOR * value

        # AI Automation
        if strategy == "AI Automation":

            if label == 1:
                return fraud_loss * AI_ERROR_RATE
            else:
                return 0

        # Human Review
        elif strategy == "Human Review":

            if label == 1:
                return HUMAN_REVIEW_COST + fraud_loss * HUMAN_ERROR_RATE
            else:
                return HUMAN_REVIEW_COST

        # Hybrid
        else:

            if row["risk_probability"] < HYBRID_THRESHOLD:

                if label == 1:
                    return fraud_loss * AI_ERROR_RATE
                else:
                    return 0

            else:

                if label == 1:
                    return HUMAN_REVIEW_COST + fraud_loss * HUMAN_ERROR_RATE
                else:
                    return HUMAN_REVIEW_COST

    sim_df["realized_cost"] = sim_df.apply(realized_cost, axis=1)

    # ---------------------------------
    # Step 4: Aggregate Metrics
    # ---------------------------------

    total_cost = sim_df["realized_cost"].sum()
    avg_cost_per_order = sim_df["realized_cost"].mean()

    strategy_dist = sim_df["optimal_strategy"].value_counts(normalize=True)

    ai_rate = strategy_dist.get("AI Automation", 0)
    human_rate = strategy_dist.get("Human Review", 0)
    hybrid_rate = strategy_dist.get("Hybrid AI + Human", 0)

    # ---------------------------------
    # Step 5: Cost Breakdown
    # ---------------------------------

    ai_df = sim_df[sim_df["optimal_strategy"] == "AI Automation"]
    human_df = sim_df[sim_df["optimal_strategy"] == "Human Review"]
    hybrid_df = sim_df[sim_df["optimal_strategy"] == "Hybrid AI + Human"]

    ai_cost = ai_df["realized_cost"].sum()
    human_cost = human_df["realized_cost"].sum()
    hybrid_cost = hybrid_df["realized_cost"].sum()

    results = {
        "total_cost": total_cost,
        "avg_cost_per_order": avg_cost_per_order,
        "ai_rate": ai_rate,
        "human_rate": human_rate,
        "hybrid_rate": hybrid_rate,
        "ai_total_cost": ai_cost,
        "human_total_cost": human_cost,
        "hybrid_total_cost": hybrid_cost
    }

    return results, sim_df

In [ ]:
# %%
# ==============================
# Run Policy Simulation
# ==============================

simulation_results, simulation_df = run_policy_simulation(decision_df)

simulation_results

# 12. AI Explanation Layer – LLM-Powered Decision Interpretation

## Business Objective

Machine learning models often produce outputs that are difficult for non-technical stakeholders to interpret.

Although our decision intelligence system already generates:

- Fraud risk probabilities
- Recommended operational strategies
- SHAP-based feature importance

these outputs still require translation into **clear business explanations** that decision makers can understand.

To address this, we introduce a **Large Language Model (LLM) explanation layer** using Google Gemini.

This layer converts structured model outputs into **concise natural language insights**, enabling analysts, operations teams, and executives to understand why a particular action was recommended.

---

## Role of the LLM in the System

The LLM **does not influence predictions or decisions**.

Instead, it acts strictly as an **interpretability layer**.

System architecture:

Model Prediction  
→ Decision Engine  
→ SHAP Feature Drivers  
→ **LLM Explanation Layer**

This design ensures:

- Deterministic model decisions
- Auditability of fraud detection logic
- Clear communication of model reasoning

---

## Production Design Principle

In real production environments, calling an LLM API for every transaction is often avoided because of:

- API rate limits
- latency
- cost
- reliability concerns

Instead, explanations are often **generated offline and cached**.

Our system therefore uses a **hybrid deployment approach**:

1. Generate explanations for sampled transactions using Gemini
2. Store explanations in a CSV artifact
3. Load explanations in the dashboard for fast display

This approach mirrors how **production fraud systems integrate LLM interpretability safely and cost-efficiently**.

In [ ]:
# ==============================
# Gemini Configuration
# ==============================

import google.generativeai as genai
import os

GEMINI_API_KEY = os.getenv("GEMINI_API_KEY")

genai.configure(api_key=GEMINI_API_KEY)

model = genai.GenerativeModel(
    "gemini-2.5-flash",
    generation_config={
        "temperature": 0.2,
        "max_output_tokens": 120
    }
)

In [ ]:
# ==============================
# Prompt Builder
# ==============================

def create_prompt(row):

    return f"""
Transaction Risk Probability: {row['risk_probability']:.2f}

Recommended Strategy: {row['optimal_strategy']}

Key Risk Drivers:
{row['top_risk_drivers']}

Explain this decision in simple business language using 3 bullet points:

• Why the transaction is risky or safe
• Which factors most influenced the decision
• What business action is recommended
"""

In [ ]:
# ==============================
# LLM Explanation Generator
# ==============================

def generate_explanation(prompt):

    try:
        response = model.generate_content(prompt)

        text = response.candidates[0].content.parts[0].text

        return text.strip()

    except Exception as e:
        return f"LLM error: {str(e)}"

In [ ]:
# ==============================
# LLM Execution Control
# ==============================

RUN_LLM = False
SAMPLE_SIZE = 5


if RUN_LLM:

    decision_sample = decision_df.sample(
        SAMPLE_SIZE,
        random_state=42
    ).copy()

    decision_sample["llm_explanation"] = [
        generate_explanation(create_prompt(row))
        for _, row in decision_sample.iterrows()
    ]

    decision_sample.to_csv(
        "llm_explanations_v2.csv",
        index=False,
        encoding="utf-8"
    )

In [ ]:
from google.colab import files

files.download("llm_explanations_v2.csv")

## Persisting Explanations for Deployment

To ensure stable deployment and avoid repeated API calls, generated explanations are stored as a dataset artifact.

Deployment workflow:

1. Generate explanations using Gemini during development
2. Save explanations to a CSV file
3. Load the file inside the Streamlit dashboard

Advantages:

- eliminates API latency
- avoids rate limits
- reduces operational cost
- ensures deterministic explanations

This approach mirrors real production systems where LLMs are used as **interpretability layers rather than real-time decision engines**.

## Persisting Explanations for Deployment

To ensure stable deployment and avoid repeated API calls, generated explanations are stored as a dataset artifact.

Deployment workflow:

1. Generate explanations using Gemini during development
2. Save explanations to a CSV file
3. Load the file inside the Streamlit dashboard

Advantages:

- eliminates API latency
- avoids rate limits
- reduces operational cost
- ensures deterministic explanations

This approach mirrors real production systems where LLMs are used as **interpretability layers rather than real-time decision engines**.

In [ ]:
import joblib

# ==============================
# Save Model
# ==============================

joblib.dump(xgb_model, "risk_model.pkl")

# ==============================
# Save Preprocessing Pipeline
# ==============================

joblib.dump(preprocess_pipeline, "preprocess_pipeline.pkl")

# ==============================
# Save Feature Columns
# ==============================

feature_columns = list(user_features)

joblib.dump(feature_columns, "feature_columns.pkl")

# ==============================
# Save Decision Configuration
# ==============================

decision_config = {
    "HYBRID_THRESHOLD": HYBRID_THRESHOLD,
    "AI_ERROR_RATE": AI_ERROR_RATE,
    "HUMAN_ERROR_RATE": HUMAN_ERROR_RATE,
    "FRAUD_COST_FACTOR": FRAUD_COST_FACTOR,
    "HUMAN_REVIEW_COST": HUMAN_REVIEW_COST
}

joblib.dump(decision_config, "decision_config.pkl")

print("✅ All deployment artifacts saved.")

In [ ]:
import joblib

# Extract column names only
feature_columns = list(user_features.columns)

joblib.dump(feature_columns, "feature_columns.pkl")

print("✅ Feature columns artifact updated correctly.")

In [ ]:
import joblib

correct_features = [
    "rating",
    "sentiment_score",
    "review_length",
    "verified_purchase",
    "helpfulness_ratio"
]

joblib.dump(correct_features, "feature_columns.pkl")

print("✅ feature_columns.pkl updated")

In [ ]:
files.download("feature_columns.pkl")

In [ ]:
model = joblib.load("risk_model.pkl")

test_input = pd.DataFrame([{
    "rating": 4,
    "sentiment_score": 0.7,
    "review_length": 120,
    "verified_purchase": 1,
    "helpfulness_ratio": 0.8
}])

prediction = model.predict_proba(test_input)

print(prediction)